# SMART AIR QUALITY & HEALTH RISK MONITORING SYSTEM
Final ML Training Notebook (Realistic & Accurate)

Based on WHO, EPA, and medical literature for CO poisoning, asthma, COPD, allergic rhinitis.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

print("🚀 Starting Final Realistic ML Training for Air Quality Health Risk System\n")

In [ ]:
# ====================== CELL 1: Generate Realistic Dataset ======================
def generate_realistic_dataset(n_samples=15000):
    np.random.seed(42)
    # Realistic sensor ranges (based on typical MQ135/MQ7 behavior)
    mq135 = np.random.randint(90, 920, n_samples)      # General air pollutants
    mq7    = np.random.randint(45, 650, n_samples)     # Carbon Monoxide
    temp   = np.random.uniform(13, 37, n_samples)      # °C
    hum    = np.random.uniform(28, 82, n_samples)      # %
    labels = []
    for i in range(n_samples):
        co = mq7[i]
        pol = mq135[i]
        t = temp[i]
        h = hum[i]
        # Real health risk rules (grounded in guidelines)
        if co > 450:                                      # High CO (approaching dangerous levels per EPA/OSHA ~50-200+ ppm equivalent)
            labels.append("CO Poisoning Risk")            # Headache, dizziness, nausea, risk of unconsciousness
        elif pol > 620 and t > 29 and h < 45:             # High pollution + hot + dry → strong asthma trigger
            labels.append("Asthma Risk")
        elif (pol > 520 or co > 300) and (t > 31 or h < 35 or h > 75):   # Multiple stressors (pollution + extreme temp/humidity)
            labels.append("High Risk")                    # COPD flare, respiratory irritation, allergic rhinitis worsening
        elif pol > 340 or co > 190:                       # Moderate exposure
            labels.append("Moderate Risk")
        else:
            labels.append("Safe")
    df = pd.DataFrame({
        'MQ135': mq135,
        'MQ7': mq7,
        'Temperature': np.round(temp, 1),
        'Humidity': np.round(hum, 1),
        'Risk': labels
    })
    print(f"✅ Generated {len(df)} realistic samples")
    print("Risk Level Distribution:")
    print(df['Risk'].value_counts())
    return df

df = generate_realistic_dataset()

In [ ]:
# ====================== CELL 2: Quick Data Overview ======================
print("\nSample Data:")
print(df.head())
print("\nAverage values by Risk Level:")
print(df.groupby('Risk').mean().round(2))

In [ ]:
# ====================== CELL 2c: Robust Data Exploration & Cleaning ======================
# 1. Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

# 2. Check for duplicates
duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")

# 3. Sort by pollutant levels and risk
print("\nTop 5 rows with highest MQ135 (pollution):")
print(df.sort_values('MQ135', ascending=False).head())

print("\nTop 5 rows with highest MQ7 (CO):")
print(df.sort_values('MQ7', ascending=False).head())

print("\nTop 5 rows with highest temperature:")
print(df.sort_values('Temperature', ascending=False).head())

print("\nTop 5 rows with lowest humidity:")
print(df.sort_values('Humidity').head())

print("\nSample rows for each risk level:")
for risk in df['Risk'].unique():
    print(f"\nRisk: {risk}")
    print(df[df['Risk'] == risk].head(2))

# 4. Correlation heatmap
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
sns.heatmap(df[['MQ135','MQ7','Temperature','Humidity']].corr(), annot=True, cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()

# 5. Class balance visualization
plt.figure(figsize=(7,4))
sns.countplot(data=df, x='Risk', order=df['Risk'].value_counts().index, palette='Set1')
plt.title('Class Balance (Risk Levels)')
plt.ylabel('Count')
plt.xlabel('Risk Level')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# 6. (Optional) Balance dataset if needed (example: upsample minority classes)
from sklearn.utils import resample

# Check class counts
class_counts = df['Risk'].value_counts()
min_count = class_counts.min()
max_count = class_counts.max()
if max_count > 1.5 * min_count:
    print("\nBalancing dataset by upsampling minority classes...")
    dfs = []
    for risk in class_counts.index:
        subset = df[df['Risk'] == risk]
        dfs.append(resample(subset, replace=True, n_samples=max_count, random_state=42))
    df_balanced = pd.concat(dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    print("Balanced class distribution:")
    print(df_balanced['Risk'].value_counts())
else:
    print("\nDataset is already reasonably balanced.")

In [ ]:
# ====================== CELL 2d: Outlier Detection & Removal ======================
# Using IQR method for each feature
def remove_outliers_iqr(df, features):
    df_clean = df.copy()
    for col in features:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        before = len(df_clean)
        df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]
        after = len(df_clean)
        print(f"Removed {before-after} outliers from {col}")
    return df_clean

features = ['MQ135', 'MQ7', 'Temperature', 'Humidity']
df_no_outliers = remove_outliers_iqr(df, features)
print(f"\nShape after outlier removal: {df_no_outliers.shape}")

# ====================== CELL 2e: Automated EDA Report (pandas-profiling) ======================
# Uncomment to use (requires pandas-profiling installed)

# from pandas_profiling import ProfileReport
# profile = ProfileReport(df_no_outliers, title='Air Quality Health Risk EDA Report', explorative=True)
# profile.to_widgets()
# profile.to_file('eda_report.html')

# ====================== CELL 2f: Model Comparison & Cross-Validation ======================
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

models = {
    'RandomForest': RandomForestClassifier(n_estimators=400, max_depth=25, random_state=42, n_jobs=-1),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'DecisionTree': DecisionTreeClassifier(max_depth=25, random_state=42)
}

X = df_no_outliers[features]
y = df_no_outliers['Risk']

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    print(f"{name} CV Accuracy: {scores.mean():.3f} ± {scores.std():.3f}")

# ====================== CELL 2g: Hyperparameter Tuning (GridSearchCV) ======================
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [15, 25],
    'min_samples_split': [2, 5]
}

gs = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_grid, cv=3, scoring='accuracy', n_jobs=-1)
gs.fit(X, y)
print(f"\nBest RandomForest params: {gs.best_params_}")
print(f"Best CV Accuracy: {gs.best_score_:.3f}")

# ====================== CELL 2h: Save Cleaned/Balanced Dataset ======================
df_no_outliers.to_csv('cleaned_air_quality_data.csv', index=False)
print("\nCleaned dataset saved as 'cleaned_air_quality_data.csv'")

In [ ]:
# ====================== CELL 2b: Visualizations ======================
import matplotlib.pyplot as plt
import seaborn as sns

# Risk Level Distribution
plt.figure(figsize=(7,4))
sns.countplot(data=df, x='Risk', order=df['Risk'].value_counts().index, palette='Set2')
plt.title('Risk Level Distribution')
plt.ylabel('Count')
plt.xlabel('Risk Level')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# Pairplot for feature relationships by Risk
sns.pairplot(df.sample(1000), hue='Risk', palette='Set2', diag_kind='kde')
plt.suptitle('Feature Relationships by Risk Level', y=1.02)
plt.show()

In [ ]:
# ====================== CELL 3: Train the Model ======================
X = df[['MQ135', 'MQ7', 'Temperature', 'Humidity']]
y = df['Risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"\nTraining samples: {len(X_train)} | Test samples: {len(X_test)}")

# Best performing model for this type of data
model = RandomForestClassifier(
    n_estimators=400, 
    max_depth=25, 
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\n✅ Training Completed!")
print(f"Test Accuracy: {accuracy:.2%}")

In [ ]:
# ====================== CELL 4: Detailed Evaluation ======================
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Feature Importance
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\n🔥 Feature Importance:")
print(importance)

In [ ]:
# ====================== CELL 5: Save Model + Test Function ======================
joblib.dump(model, 'air_quality_health_risk_model.pkl')
print("\n💾 Model saved successfully as 'air_quality_health_risk_model.pkl'")

def predict_health_risk(mq135, mq7, temperature, humidity):
    """Easy function to predict risk"""
    loaded_model = joblib.load('air_quality_health_risk_model.pkl')
    input_df = pd.DataFrame([[mq135, mq7, temperature, humidity]],
                            columns=['MQ135', 'MQ7', 'Temperature', 'Humidity'])
    risk = loaded_model.predict(input_df)[0]
    return risk

# Test predictions
print("\n🧪 Quick Test Predictions:")
print("Test 1 (Clean air)    :", predict_health_risk(180, 90, 24.5, 55))
print("Test 2 (Moderate)     :", predict_health_risk(450, 220, 28.0, 48))
print("Test 3 (Asthma risk)  :", predict_health_risk(680, 180, 31.5, 38))
print("Test 4 (CO danger)    :", predict_health_risk(520, 520, 26.0, 52))

In [ ]:
# ====================== CELL 6: Model Save/Load & Integrity Check ======================
import os

# Check if model file exists
model_path = 'air_quality_health_risk_model.pkl'
if os.path.exists(model_path):
    print(f"✅ Model file '{model_path}' exists.")
    # Try loading the model
    try:
        loaded_model = joblib.load(model_path)
        print("✅ Model loaded successfully.")
        # Optionally, test a prediction
        test_pred = loaded_model.predict([[400, 200, 25, 50]])
        print(f"Test prediction output: {test_pred}")
    except Exception as e:
        print(f"❌ Error loading model: {e}")
else:
    print(f"❌ Model file '{model_path}' does not exist. Please run the training cell first.")

# Check if cleaned dataset was saved
csv_path = 'cleaned_air_quality_data.csv'
if os.path.exists(csv_path):
    print(f"✅ Cleaned dataset '{csv_path}' exists.")
    # Optionally, load and preview
    df_check = pd.read_csv(csv_path)
    print(df_check.head())
else:
    print(f"❌ Cleaned dataset '{csv_path}' not found. Run the cleaning cell to save it.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ Notebook Ready - Smart Air Quality Health Risk Model")

In [ ]:
# Generate realistic synthetic data based on actual health risks
def generate_realistic_dataset(n_samples=12000):
    np.random.seed(42)
    mq135 = np.random.randint(80, 950, n_samples)      # Air quality / gases
    mq7    = np.random.randint(40, 680, n_samples)     # Carbon Monoxide
    temp   = np.random.uniform(12, 38, n_samples)      # Temperature °C
    hum    = np.random.uniform(25, 85, n_samples)      # Humidity %
    labels = []
    for i in range(n_samples):
        co = mq7[i]
        pol = mq135[i]
        t = temp[i]
        h = hum[i]
        # Real-world health risk mapping
        if co > 480:                                   # High CO exposure → CO Poisoning
            labels.append("CO Poisoning Risk")
        elif pol > 650 and t > 28 and h < 48:          # High pollution + hot + dry air → Asthma attack trigger
            labels.append("Asthma Risk")
        elif (pol > 520 or co > 320) and (t > 30 or h < 35 or h > 75):  # Multiple stressors → High Risk
            labels.append("High Risk")
        elif pol > 320 or co > 180:                    # Moderate exposure
            labels.append("Moderate Risk")
        else:
            labels.append("Safe")
    df = pd.DataFrame({
        'MQ135': mq135,
        'MQ7': mq7,
        'Temperature': np.round(temp, 1),
        'Humidity': np.round(hum, 1),
        'Risk': labels
    })
    return df
df = generate_realistic_dataset()
print("Dataset Shape:", df.shape)
print("\nRisk Distribution:\n", df['Risk'].value_counts())
df.head()

In [ ]:
print("Basic Statistics")
print(df.describe())
print("\nAverage values per Risk Level:")
print(df.groupby('Risk').mean().round(2))
# Optional visualization (run if you have matplotlib)
# sns.countplot(data=df, x='Risk')
# plt.title('Risk Level Distribution')
# plt.show()

In [ ]:
X = df[['MQ135', 'MQ7', 'Temperature', 'Humidity']]
y = df['Risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")
# Train best model (Random Forest)
model = RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\n✅ Model Trained Successfully!")
print(f"Test Accuracy: {accuracy:.2%}")

In [ ]:
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, digits=4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)
print("\nFeature Importance:")
print(importance)

In [ ]:
# Save the trained model
joblib.dump(model, 'air_quality_health_risk_model.pkl')
print("✅ Model saved as 'air_quality_health_risk_model.pkl'")
# Reusable prediction function
def predict_health_risk(mq135, mq7, temperature, humidity):
    loaded_model = joblib.load('air_quality_health_risk_model.pkl')
    input_data = pd.DataFrame([[mq135, mq7, temperature, humidity]],
                              columns=['MQ135', 'MQ7', 'Temperature', 'Humidity'])
    risk = loaded_model.predict(input_data)[0]
    return risk
# Test the function
print("\nTest Prediction:")
print(predict_health_risk(420, 250, 29.5, 42))